In [2]:
import feast
import os
from loguru import logger
import dagshub
import mlflow
import time
import joblib
from datetime import datetime
import uuid
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from feast import (
    FeatureStore,
    Entity,
    FeatureService,
    FeatureView,
    Field,
    FileSource
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from feast.types import Float32, Float64, Int64, String

import reescalador

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import (GridSearchCV,
                                     RandomizedSearchCV,
                                     train_test_split)
from sklearn import svm
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (classification_report,
                            confusion_matrix,
                            ConfusionMatrixDisplay,
                            accuracy_score)

from xgboost import XGBClassifier, plot_importance
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


dagshub.init(repo_owner='Marquinhos9873', repo_name='mle3', mlflow=True)



Accessing as Marquinhos9873

Initialized MLflow to track repo "Marquinhos9873/mle3"

Repository Marquinhos9873/mle3 initialized!

#### Set tracking // Params

In [13]:
params={"Decision Tree": {
                        "criterion": ['gini','log_loss'],
                        "max_depth": [0, 10, 12, 16],
                        "min_samples_split": [2, 3, 4, 5]
                    },
    
                     "Random Forest":{
                        "n_estimators" : [110 , 115, 125, 130],
                        "criterion" : ['gini' ,'log_loss', 'entropy'],
                        "max_depth" : [None, 5, 7 , 8]
                    },
    
                    "Gradient Boosting": {
                        "n_estimators" : [110 , 115, 125, 130],
                        "max_depth" : [None, 5, 7 , 8],
                        "learning_rate": [0.0045 , 0.01, 0.05, 0.10],
                        "min_samples_split": [2, 3, 4, 5],
                    },
                    "XGBClassifier":{
                        'learning_rate':[0.0045 , 0.01, 0.05, 0.10],
                        'max_depth': [5, 6, 7],
                        'n_estimators': [256, 128, 64, 12],
                        'tree_method': ['auto', 'approx']
                    },
    
                    "CatBoosting Classifier":{
                        'iterations': [200, 400, 800],
                        'learning_rate': [0.001, 0.01, 0.05, 0.1],
                        'depth': [3, 4, 6, 8, 10],
                        'l2_leaf_reg': [1, 3, 5, 7, 9],
                        'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS']
                    }
                    
                }

In [4]:
SAVE_DIR = "/home/marcoloq/mle2/data/model_metrics_local_prueba_2/"
os.makedirs(SAVE_DIR, exist_ok=True)


In [15]:
xgb = XGBClassifier(device= 'cuda', verbosity = 1)

In [16]:
kitty = CatBoostClassifier(task_type = 'GPU', early_stopping_rounds = 100)

In [17]:
Forest = RandomForestClassifier(verbose = 1)

In [18]:
Grades = GradientBoostingClassifier(verbose = 1) 

In [7]:
Ada = AdaBoostClassifier()

In [10]:
machine = SVC()

#### Def calculate_clsf_model

In [67]:
def evaluate_classification_model(name_model: str, y_real, predictions, probabilities, save_path: str = None):
    
    
    label_map = {
        0: "No Stress",
        1: "Distress",
        2: "Eustress"
    }
    display_labels = [label_map[i] for i in sorted(label_map.keys())]

    
    cm = confusion_matrix(y_real, predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)

    plt.figure(figsize=(6, 5))
    ax = plt.gca()                     
    disp.plot(cmap="Blues", values_format='d', ax=ax)
    plt.title(f"Matriz de Confusión - {name_model}")
    plt.tight_layout()

    if save_path:
        plt.savefig(f"{save_path}/{name_model}_confusion_matrix.png")

    plt.show()


    
    accuracy = accuracy_score(y_real, predictions)
    precision = precision_score(y_real, predictions, average='weighted')
    recall = recall_score(y_real, predictions, average='weighted')
    f1 = f1_score(y_real, predictions, average='weighted')

    
    try:
        auc_score = roc_auc_score(y_real, probabilities, multi_class='ovr')
    except:
        auc_score = None
        print("AUC no disponible para este modelo.")

    print(f"\n>>> Métricas del modelo: {name_model}")
    print(f"accuracy : {accuracy}")
    print(f"precision: {precision}")
    print(f"recall   : {recall}")
    print(f"f1_score : {f1}")
    print(f"auc_score: {auc_score}")

    metricas = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "auc_score": auc_score
    }



    return metricas
